# 00 — Preliminary Analysis

**Objective.** Audit the raw dataset before anything is decided about it. This notebook establishes
what the data actually contains — types, duplicates, gaps, distributions, extreme values and
structural quirks — so that every treatment applied later in `01_global_preprocessing.ipynb` can
point at a specific result here as its justification.

**Input:** `data/raw/SupplyFlow FMCG Solutions.xlsx`
**Output:** none. *This notebook reports only.* It applies no treatment, imputes nothing, removes
nothing, and writes no data. That is deliberate — investigation and treatment are kept apart so the
reasoning behind each treatment stays visible.

**What it closes:** nothing. Every section ends with an **open decision** that
`01_global_preprocessing.ipynb` resolves, using the evidence produced here.

---

### Sections

| § | Question |
|---|---|
| 1 | How big is the dataset and what does a row look like? |
| 2 | Is each field stored in a type that matches its business definition? |
| 3 | Are there duplicate records, and are the identifiers really unique? |
| 4 | Where are values missing — including values that represent missingness without being blank? |
| 5 | Is each gap random, or does it carry information? |
| 6 | How is each numeric column distributed? |
| 7 | How is each categorical column distributed? |
| 8 | Which values are extreme, and are they errors or real? |
| 9 | What shape do the two target variables take? |
| 10 | Does any column reproduce or constrain another? |
| 11 | Which numeric columns move together? |
| 12 | Confirmation that nothing was modified. |


## 0. Setup


In [ ]:
import sys, pathlib

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *

set_style()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Snapshot of data/preprocessed/ before this notebook runs, so §12 can prove it wrote nothing there.
preprocessed_at_start = {p.name: p.stat().st_mtime for p in PREPROCESSED_DIR.glob('*')}

print("project root :", PROJECT_ROOT)
print("source file  :", RAW_FILE.name)
print("pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", sns.__version__)

---
## 1. Load and shape

The supplied specification states 25,000 rows and 24 columns. First check is simply whether the delivered file
matches that description.


In [ ]:
df = load_raw()

print(f"rows    : {df.shape[0]:,}")
print(f"columns : {df.shape[1]}")
print(f"memory  : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
df.head()

In [ ]:
df.info()

> **Interpretation.**
>
> - The delivered file matches the supplied specification exactly: **25,000 rows and 24
>   columns**, one row per warehouse, 13.55 MB in memory.
>
> - `df.info()` shows three columns are not fully populated — `workers_num` (24,010 non-null),
>   `wh_est_year` (13,119) and `approved_wh_govt_certificate` (24,092). Everything else has all
>   25,000 values. Those gaps are quantified in §4.
>
> - The dtype split is 8 text, 14 integer, 2 float. Note that `workers_num` and `wh_est_year` are the
>   two floats, and both are conceptually whole numbers — a column of integers is forced to float the
>   moment it contains a missing value, so the float dtype here is a *symptom of the gaps*, not a
>   separate problem.


---
## 2. Data type check

First preparation step: confirm each field is stored in the correct format.

The `expected` column below is transcribed from `docs/01_data_dictionary.md`, which was written
from the supplied business definitions **before** the data was opened. This cell compares that
expectation against what pandas actually inferred. A mismatch is not automatically a defect — it is
a question to answer.


In [ ]:
expected_type = {
    "Ware_house_ID": "identifier",              "WH_Manager_ID": "identifier",
    "Location_type": "nominal",                 "WH_capacity_size": "ordinal",
    "zone": "nominal",                          "WH_regional_zone": "nominal",
    "num_refill_req_l3m": "count",              "transport_issue_l1y": "count",
    "Competitor_in_mkt": "count",               "retail_shop_num": "count",
    "wh_owner_type": "nominal",                 "distributor_num": "count",
    "flood_impacted": "binary",                 "flood_proof": "binary",
    "electric_supply": "binary",                "dist_from_hub": "continuous",
    "workers_num": "count",                     "wh_est_year": "year",
    "storage_issue_reported_l3m": "count",      "temp_reg_mach": "binary",
    "approved_wh_govt_certificate": "ordinal",  "wh_breakdown_l3m": "count",
    "govt_check_l3m": "count",                  "product_wg_ton": "continuous",
}

type_check = pd.DataFrame({
    "pandas_dtype": df.dtypes.astype(str),
    "expected": pd.Series(expected_type),
    "n_unique": df.nunique(),
    "n_missing": df.isna().sum(),
})
type_check["example_values"] = [
    ", ".join(map(str, df[c].dropna().unique()[:4])) for c in df.columns
]
type_check

> **Interpretation.**
>
> - Every field is stored in a type consistent with its business definition.
>   Nothing needs recasting before it can be used, and the data-type check passes.
>
> - Four observations worth carrying forward:
>   - **`workers_num` (count) and `wh_est_year` (year) are `float64`.** As noted in §1 this follows
>     from their missing values, not from any decimal content. If the gaps are filled they can return
>     to integer.
>   - **Both identifier columns hold 25,000 distinct values** — one per row. They identify records
>     rather than describe them.
>   - **`WH_capacity_size` and `approved_wh_govt_certificate` arrive as unordered text**, but the data
>     dictionary records both as ordinal (Small < Mid < Large; C < B < B+ < A < A+). Pandas cannot
>     know that ordering — it has to be imposed deliberately at encoding time, and doing so is what
>     distinguishes ordinal encoding from label encoding here.
>   - **`approved_wh_govt_certificate` reports 5 levels and 908 missing.** So whatever represents the
>     missing state is *not* one of the five grades. §4 establishes what it actually is.

> **Decision opened.**
>
> - Decide whether the two identifier columns should be dropped (§3 supplies the evidence).
> - Decide whether `workers_num` and `wh_est_year` should return to integer type after their gaps are treated.
> - Both are resolved in `01_global_preprocessing.ipynb`.


---
## 3. Structural integrity

The structural audit checks duplicates in two ways:

- **Exact duplicate rows** — the same record entered twice.
- **Duplicate rows once the identifier columns are ignored** — two warehouses with different IDs but
  identical attributes. That would be worth knowing before treating each row as an independent
  observation.

Cardinality is checked at the same time, because a column with one distinct value per row is an
identifier rather than a feature.


In [ ]:
id_cols = ["Ware_house_ID", "WH_Manager_ID"]

print(f"exact duplicate rows                   : {df.duplicated().sum():,}")
print(f"duplicate rows ignoring the ID columns : {df.drop(columns=id_cols).duplicated().sum():,}")
print()
for c in id_cols:
    unique = df[c].nunique()
    verdict = "one per row" if unique == len(df) else "NOT one per row"
    print(f"{c:<16} {unique:,} distinct values across {len(df):,} rows  ->  {verdict}")

In [ ]:
pd.DataFrame({
    "n_unique": df.nunique(),
    "pct_of_rows": (100 * df.nunique() / len(df)).round(3),
}).sort_values("n_unique", ascending=False)

> **Interpretation.**
>
> - Duplicate removal has
>   **nothing to remove**: 0 exact duplicate rows, and 0 duplicates even after ignoring the two
>   identifier columns. The second check is the more demanding one — it asked whether any two
>   warehouses share an identical set of 22 attributes, and none do. The duplication step is therefore
>   recorded as a **check that passed**, not a step skipped.
>
> - Both identifiers have 25,000 distinct values across 25,000 rows — 100% cardinality, exactly one
>   per row. They carry no information about how a warehouse operates. Leaving them in would let a
>   tree-based model split on row identity, which fits the training set perfectly and generalises to
>   nothing.
>
> - The cardinality table also confirms nothing else is accidentally an identifier. The next highest
>   are `retail_shop_num` at 4,906 distinct values (19.62% of rows) and `product_wg_ton` at 4,561
>   (18.24%) — high, but that is simply what a continuous measurement over 25,000 warehouses looks
>   like, and both are genuine quantities.

> **Decision opened.**
>
> - Drop `Ware_house_ID` as a feature, while keeping it as the row key.
> - Drop `WH_Manager_ID` completely.
> - Remove no rows for duplication.
> - Apply this in `01_global_preprocessing.ipynb`.


---
## 4. Missing values

Four passes, because a blank cell is only the most obvious way a value can be absent.

1. **Nulls after a normal read** — what `pandas` reports as missing.
2. **Text that stands for missing** — pandas silently converts a fixed list of strings (`"NA"`,
   `"N/A"`, `"null"`, `"NaN"`, and others) to `NaN` while reading. So a cell that literally contains
   the text `NA` and a cell that is genuinely empty become indistinguishable after loading. Reading
   the file a second time with that conversion switched off separates the two. The distinction
   matters: an empty cell is an absent measurement, whereas a deliberate `NA` may be a recorded
   state.
3. **Numeric sentinels** — negatives where the definition allows none, and zeros in columns where a
   zero may be a real reading or may be a placeholder.
4. **Where in the file the gaps sit** — spread evenly through the rows, or concentrated in one
   stretch? A block of consecutive gaps would point to a data-entry or export fault rather than to
   anything about the warehouses themselves.


In [ ]:
missing = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": (100 * df.isna().mean()).round(2),
})
missing = missing[missing["n_missing"] > 0].sort_values("pct_missing", ascending=False)

print(f"columns with at least one null: {len(missing)} of {df.shape[1]}")
missing

> **Interpretation.**
>
> The null scan finds gaps in exactly three of the 24 columns: `wh_est_year` at nearly half the rows (47.52%), `workers_num` at under 4%, and `approved_wh_govt_certificate` at 3.63%. The remaining 21 columns are complete.

In [ ]:
# Pass 2 - what do those cells literally contain in the file?
raw_text = pd.read_excel(RAW_FILE, sheet_name="Data", keep_default_na=False, dtype=str)

sentinel_rows = []
for c in df.columns:
    if df[c].isna().sum() == 0:
        continue
    for value, n in raw_text.loc[df[c].isna(), c].value_counts().items():
        sentinel_rows.append({
            "column": c,
            "literal_cell_content": "(truly empty cell)" if value == "" else repr(value),
            "n_rows": int(n),
            "pct_of_rows": round(100 * n / len(df), 2),
        })
pd.DataFrame(sentinel_rows)

> **Interpretation.**
>
> Pass 2 reveals that the 908 apparent nulls in `approved_wh_govt_certificate` are not blanks but cells that contain the literal text `'NA'`. The column therefore has no true gap; it has a state that the standard Excel reader converts to missing. The two genuine gap columns — `wh_est_year` and `workers_num` — show only blank cells.

In [ ]:
# Pass 3 - numeric sentinels.
num_cols = list(df.select_dtypes("number").columns)

pd.DataFrame({
    "min": [df[c].min() for c in num_cols],
    "n_negative": [int((df[c] < 0).sum()) for c in num_cols],
    "n_zero": [int((df[c] == 0).sum()) for c in num_cols],
    "pct_zero": [round(100 * (df[c] == 0).mean(), 2) for c in num_cols],
}, index=num_cols).sort_values("pct_zero", ascending=False)

> **Interpretation.**
>
> The sentinel scan finds no negative values and no zero values that look like coded absences. Zeros in `flood_impacted`, `flood_proof`, `wh_breakdown_l3m` and similar columns are genuine count values, not placeholders.

In [ ]:
plt.figure(figsize=(11, 4))
sns.heatmap(df.isna(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Missing value map  (light = missing)", fontweight="bold")
plt.xlabel("column"); plt.ylabel("rows, in file order")
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> The heatmap shows the gaps in `wh_est_year` as a diffuse band running the full height of the column with no visible clustering at any part of the file. The two smaller gaps in `workers_num` and `approved_wh_govt_certificate` are barely visible at this scale, consistent with their low rates.

In [ ]:
# Pass 4 - missing rate in consecutive blocks of 2,500 rows, in file order.
# The map above compresses 25,000 rows into a few hundred pixels, so scattered
# gaps in the smaller columns barely render. This table does not have that problem.
block = pd.Series(np.arange(len(df)) // 2500 + 1, index=df.index, name="rows_block")
(100 * df[missing.index].isna().groupby(block).mean()).round(2)

> **Interpretation.**
>
> The block table confirms that each of the three columns has a roughly stable gap rate across all ten row-blocks; no single block stands out as the source of most gaps. The spread through the file is uniform rather than batch-concentrated.

> **Interpretation.**
>
> - Only **3 of 24 columns** have any gap, and they differ enormously in scale:
>   `wh_est_year` 11,881 rows (**47.52%**), `workers_num` 990 (3.96%), and
>   `approved_wh_govt_certificate` 908 (3.63%).
>
> - **Pass 2 shows the gaps are not all the same kind.**
>
> | Column | What the cell actually contains |
> |---|---|
> | `workers_num` | a truly empty cell — 990 rows |
> | `wh_est_year` | a truly empty cell — 11,881 rows |
> | `approved_wh_govt_certificate` | the **literal text `'NA'`** — all 908 rows |
>
> - Every one of the 908 certificate gaps is a cell where someone wrote `NA`. Pandas converted that
>   text to a null while reading, which made it look identical to a blank. It is not: a blank is an
>   **absent measurement**, while a written `NA` is a **recorded state**. Filling it with a grade would
>   invent a certificate the file says was never issued.
>
> - **Pass 3** finds no negative values, so there are no `-1`-style sentinels. Zeros are common only
>   where zero simply means "no" — the 0/1 indicators (`flood_proof` 94.54%, `flood_impacted` 90.18%,
>   `temp_reg_mach` 69.67%, `electric_supply` 34.31%) and `transport_issue_l1y` (60.86%).
>   One coincidence is followed up in §10: **`storage_issue_reported_l3m` and `wh_breakdown_l3m` each
>   have exactly 908 zeros** — the same count as the certificate's `NA`.
>
> - **Pass 4 shows the gaps are spread evenly through the file.** Across ten consecutive blocks of
>   2,500 rows the missing rate stays between 45.48% and 49.56% for `wh_est_year`, 3.72% and 4.16%
>   for `workers_num`, and 2.88% and 4.16% for the certificate. No stretch of rows is responsible, which
>   rules out a fault in one batch of records — the explanation has to be looked for in the warehouses
>   themselves, which is what §5 does.
>
> - The missing-value map agrees for `wh_est_year`, whose gaps appear as stripes from top to bottom.
>   For the two smaller columns the map is not informative: 25,000 rows are squeezed into a few hundred
>   pixels, so gaps affecting about 4% of scattered rows barely render. That limitation is why Pass 4
>   was added.


---
## 5. Missingness mechanism

Possible treatments are removal or imputation. Extent alone is not enough to choose between them — the **reason** for the gap matters more:

- If a value is missing **completely at random**, the rows that lost it are otherwise a fair sample.
  Imputation introduces no bias, and removal only costs sample size.
- If missingness is **related to other recorded values**, or to an outcome, the affected rows are
  not a fair sample. Removing them would bias the analysis, and imputing without recording that a
  value was missing would throw away real information.

Two tests distinguish these:

- **Test A** — is the missing rate roughly equal across the levels of every categorical column?
- **Test B** — do the rows with a gap differ from the rows without one on **every other numeric
  column**, the two targets included? Mann–Whitney supplies the p-value because it assumes nothing
  about distribution shape.

With 25,000 rows almost any difference produces a small p-value, so the p-value alone cannot say
whether a difference *matters*. Test B therefore also reports the **standardised mean difference
(SMD)** — the gap between the two group means divided by their pooled standard deviation, read as
"how many standard deviations apart". A common rule of thumb treats |SMD| ≥ 0.1 as a difference
worth attention.


In [ ]:
missing_cols = list(missing.index)
cat_cols = [c for c in df.select_dtypes("str").columns if c not in id_cols]
targets = ["product_wg_ton", "wh_breakdown_l3m"]

print("columns with gaps to explain :", missing_cols)
print("categoricals to test against :", cat_cols)
print("targets                      :", targets)

In [ ]:
# Test A - missing rate across the levels of each categorical.
for col in missing_cols:
    print(f"\n=== missing rate of '{col}' by category ===")
    flagged = df.assign(_missing=df[col].isna())
    for cat in cat_cols:
        rate = 100 * flagged.groupby(cat, dropna=False)["_missing"].mean()
        print(f"  {cat:<30} {rate.min():5.2f}%  to {rate.max():5.2f}%"
              f"   (spread {rate.max() - rate.min():5.2f} percentage points)")

In [ ]:
# Test B - do rows with a gap differ from rows without one, on every other numeric column?
rows = []
for col in missing_cols:
    gap = df[col].isna()
    for other in num_cols:
        if other == col:
            continue
        a = df.loc[gap, other].dropna()
        b = df.loc[~gap, other].dropna()
        pooled_sd = np.sqrt((a.var() + b.var()) / 2)
        _, p = stats.mannwhitneyu(a, b)
        rows.append({
            "gap_in": col,
            "compared_on": other,
            "mean_when_missing": round(a.mean(), 2),
            "mean_when_present": round(b.mean(), 2),
            "smd": round((a.mean() - b.mean()) / pooled_sd, 3),
            "mannwhitney_p": f"{p:.3g}",
        })
test_b = pd.DataFrame(rows)
test_b["abs_smd"] = test_b["smd"].abs()

for col in missing_cols:
    sub = test_b[test_b["gap_in"] == col].sort_values("abs_smd", ascending=False)
    n_big = int((sub["abs_smd"] >= 0.1).sum())
    print(f"\n=== gap in '{col}':  {n_big} of {len(sub)} columns differ by |SMD| >= 0.1 "
          f"— six largest shown ===")
    print(sub.head(6).drop(columns=["gap_in", "abs_smd"]).to_string(index=False))

> **Interpretation.**
>
> - Extent alone is the wrong criterion. The three gaps are three different
>   problems, and **none of them is purely random**.
>
> - **`wh_est_year` (47.52% missing) — unrelated to the categoricals, strongly related to operating
>   activity.**
>   Test A: the missing rate barely moves across any categorical (spread between 0.57 and 3.52
>   percentage points). Test B tells a different story — **5 of 15 numeric columns differ by
>   |SMD| ≥ 0.1**, one of them enormously:
>
> | Compared on | Year missing | Year present | SMD |
> |---|---|---|---|
> | `num_refill_req_l3m` | 2.55 | 5.49 | **−1.353** |
> | `transport_issue_l1y` | 1.13 | 0.45 | +0.589 |
> | `temp_reg_mach` | 0.20 | 0.40 | −0.451 |
> | `product_wg_ton` | 20,100.85 | 23,915.51 | −0.334 |
> | `storage_issue_reported_l3m` | 15.80 | 18.33 | −0.279 |
>
> - Warehouses with no recorded year average **fewer than half the refill requests** of those with one
>   — a gap of more than one full standard deviation. They also report more transport issues, less often
>   have temperature regulation, ship less, and report fewer storage issues. So the rows missing a year
>   have a **distinguishable operating profile**. Two consequences follow. Dropping them would remove
>   47.52% of the data *and* a systematically different kind of warehouse, biasing every objective. And
>   whatever value is filled in, the model should be able to see that it was filled, because the fact
>   of missingness itself carries information.
>
> - **`workers_num` (3.96% missing) — related to site conditions, not to outcomes.**
>   Test A is flat (spread 0.20 to 1.66 pp). Test B finds **3 of 15 columns** differing by
>   |SMD| ≥ 0.1, all of them site indicators: rows missing a worker count are more often
>   flood-impacted (0.23 vs 0.09, SMD +0.391), more often flood-proof (0.16 vs 0.05, +0.368), and more
>   often have electric back-up (0.74 vs 0.65, +0.190). Beyond those three, the largest difference
>   anywhere is |SMD| 0.045 — so neither target differs meaningfully.
>   Had the gap been compared only against the two targets, it would have looked completely random.
>   Comparing it against every column shows it is not. That matters for how it is filled: §11 shows
>   `workers_num` itself correlates with `electric_supply` (0.340) and `flood_impacted` (0.168), so an
>   overall median may sit slightly off for exactly the rows being filled. Whether the difference is
>   large enough to matter is measured in `01_global_preprocessing.ipynb`, not assumed.
>
> - **`approved_wh_govt_certificate` (3.63%) — not a gap at all.**
>   Test B finds 6 of 16 columns differing, by margins far beyond anything else in this notebook:
>   breakdowns **0.00 vs 3.61** (SMD −3.238), storage issues **0.00 vs 17.78** (−2.891), establishment
>   year **2,021.94 vs 2,008.91** (+2.522), shipment **5,430.47 vs 22,730.99 t** (−2.099), and
>   temperature regulation 0.01 vs 0.31 (−0.890). Differences of two to three standard deviations mean
>   the two groups barely overlap. Together with §4's finding that the cells literally read `NA`, this is
>   a distinct subgroup of warehouses rather than missing data. §10 identifies it.
>
> - One artefact to discount: the 100.00 pp spread of the certificate's missing rate "by
>   `approved_wh_govt_certificate`" is the column tested against its own levels. The informative row is
>   `Location_type`, where the rate runs **0.00% to 3.96%** — one of the two location types contains no
>   unrated warehouses at all.

> **Decision opened.**
> - `wh_est_year` — keep the rows; fill the gap, and record which values were filled.
> - `workers_num` — fill the gap; whether one overall value is good enough, or the flood and
>   electric groups need their own, is measured in NB 01.
> - Certificate `NA` — treat as a category in its own right, not as a gap to fill.
>
>
> - All three are resolved in `01_global_preprocessing.ipynb`.


---
## 6. Univariate analysis — numeric columns

Descriptive statistics, then the shape of each distribution. Skewness and kurtosis are reported
because they, not habit, inform whether a transformation is worth applying later — but a summary
statistic can hide shape, so the histograms are read as well, not just the table.

Whole-number columns with up to 60 distinct values are drawn with **one bar per value**. Binning
them into a fixed number of bins would produce empty or merged bars that come from the bin width,
not from the data.


In [ ]:
num_summary = df[num_cols].describe().T
num_summary["skew"] = df[num_cols].skew()
num_summary["kurtosis"] = df[num_cols].kurtosis()
num_summary.round(3)

> **Interpretation.**
>
> The summary table shows that most numeric columns are centred near whole-number values with moderate spread. `wh_est_year` spans 1996–2023 and has the largest standard deviation in calendar terms. `product_wg_ton` and `retail_shop_num` have the widest operational ranges. Skewness is low for most columns, though `workers_num` at 1.060 is the most right-skewed.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for ax, c in zip(axes.ravel(), num_cols):
    s = df[c].dropna()
    if bool((s % 1 == 0).all()) and s.nunique() <= 60:
        sns.histplot(s, discrete=True, ax=ax)
    else:
        sns.histplot(s, bins=30, ax=ax)
    ax.set_title(c, fontsize=9)
    ax.set_xlabel(""); ax.set_ylabel("")
for ax in axes.ravel()[len(num_cols):]:
    ax.axis("off")
fig.suptitle("Univariate distributions — numeric columns", fontweight="bold", y=1.001)
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> The histograms reveal shapes that summary statistics cannot capture. `product_wg_ton` is multi-modal rather than smooth despite its modest skew. `storage_issue_reported_l3m` jumps from zero straight to four, leaving a visible gap at 1–3. `govt_check_l3m` is heavily spiked at low values. Most other columns are approximately uniform or roughly bell-shaped.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 10))
for ax, c in zip(axes.ravel(), num_cols):
    sns.boxplot(x=df[c], ax=ax)
    ax.set_title(c, fontsize=9); ax.set_xlabel("")
for ax in axes.ravel()[len(num_cols):]:
    ax.axis("off")
fig.suptitle("Boxplots — numeric columns", fontweight="bold", y=1.001)
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> - The table says nothing here is severely skewed. The histograms add detail the
>   table cannot show, and in two cases change the picture.
>
> - **Near-uniform — no central tendency.** `dist_from_hub` (skew −0.006, kurtosis −1.201),
>   `distributor_num` (0.015, −1.188), `num_refill_req_l3m` (−0.075, −1.221) and `wh_est_year`
>   (0.012, −1.176) combine near-zero skew with kurtosis close to −1.2, the signature of a flat
>   distribution, and the histograms confirm it: each value is roughly as common as any other. Two
>   small departures are visible in the plots: the refill value 2 is noticeably less frequent than the
>   other eight values, and `wh_est_year` thins out at both ends (1996–1997 and 2022–2023). For these
>   columns the mean does not describe a typical warehouse, because there is no typical value — for
>   example, warehouses are spread evenly from 55 to 271 units from their hub rather than gathered
>   around the mean of 163.5. (The source file gives no unit for this distance.)
>
> - **Single peak, long right tail.** `workers_num` (skew 1.060; median 28, Q3 33, maximum 98),
>   `retail_shop_num` (0.908), `Competitor_in_mkt` (0.978) and `transport_issue_l1y` (1.611, most
>   warehouses reporting none). A few warehouses sit far above the rest; §8 quantifies them.
>
> - **0/1 indicators.** `flood_proof` (skew 3.919) and `flood_impacted` (2.701) look highly skewed, but
>   for a 0/1 column skew only restates how rare the 1s are (means 0.055 and 0.098). They are not
>   candidates for a transformation.
>
> - **Irregular — shapes the table hides entirely.**
>   - `storage_issue_reported_l3m` has an isolated bar at 0, then **no warehouses at all** until the
>     values resume, then a jagged spread. §10 scan D prints the exact counts.
>   - `govt_check_l3m` is strongly spiked: some single values are several times as common as their
>     immediate neighbours, and others are rare. Because each bar here is one value rather than a
>     bin, this is a property of the data and not of the plotting.
>
> - For both, a mean or median describes very few actual warehouses. That is worth remembering when
>   cluster profiles and group comparisons are read later.
>
> - **The Objective 3 target is not as simple as its statistics suggest.** `product_wg_ton` has skew
>   0.332 and a mean and median within one ton of each other, yet its histogram has more than one peak.
>   §9 examines it.
>
> - The boxplots show the same extremes that §8 quantifies. **No column's shape calls for a
>   transformation at this audit stage.**


---
## 7. Univariate analysis — categorical columns

Frequencies for every categorical, with missing shown as its own level rather than hidden. A level
that occupies a very small share of the rows is worth noting now, because it will end up thinly
represented in any train/test split.


In [ ]:
for c in cat_cols:
    vc = df[c].value_counts(dropna=False)
    summary = pd.DataFrame({"n": vc, "pct_of_rows": (100 * vc / len(df)).round(2)})
    print(f"\n=== {c}  ({df[c].nunique()} non-null levels) ===")
    print(summary.to_string())

> **Interpretation.**
>
> The frequency tables show pronounced imbalances in two columns. `Location_type` runs 91.83% Rural against 8.17% Urban. `zone == East` holds only 1.72% of warehouses — far below the other four zones. The five certificate grades are broadly flat, with the literal `'NA'` entries forming a small additional group.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, c in zip(axes.ravel(), cat_cols):
    # Missing is shown as its own bar rather than dropped, so a gap is visible
    # on the chart instead of silently shrinking the column.
    plot_col = df[c].fillna("(missing)")
    sns.countplot(x=plot_col, order=plot_col.value_counts().index, ax=ax)
    ax.set_title(c, fontsize=10); ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
for ax in axes.ravel()[len(cat_cols):]:
    ax.axis("off")
fig.suptitle("Category frequencies  ((missing) shown as its own level)",
             fontweight="bold", y=1.001)
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> The bar charts confirm the imbalances and add a visual anchor for each column's relative proportions. The Rural dominance in `Location_type` and the thinness of `zone == East` are immediately visible. Missing values are plotted as a distinct bar rather than dropped, making any gap visible at a glance.

In [ ]:
# Levels holding less than 5% of the rows.
rare = []
for c in cat_cols:
    for level, n in df[c].value_counts(dropna=False).items():
        pct = 100 * n / len(df)
        if pct < 5:
            rare.append({
                "column": c,
                "level": "(missing)" if pd.isna(level) else level,
                "n": int(n),
                "pct_of_rows": round(pct, 2),
            })
pd.DataFrame(rare) if rare else "No categorical level holds less than 5% of the rows."

> **Interpretation.**
>
> - The network is far from evenly distributed, and two imbalances matter.
>
> - **Location is heavily skewed toward Rural — 91.83% against 8.17% Urban.** Any comparison between
>   the two rests on 2,043 urban warehouses; enough to estimate, but the imbalance must be stated
>   whenever location is discussed.
>
> - **`zone` contains one very thin level: East, at 429 rows (1.72%).** North (41.11%), West (31.72%)
>   and South (25.45%) between them hold 98.28% of the network. This is the main practical concern in
>   this section. In a stratified 80/20 split East contributes roughly 86 warehouses to the test set —
>   workable for a main effect, but too thin to support any claim about how East behaves in
>   combination with another feature.
>
> - The remaining categoricals are comfortable. `WH_capacity_size` runs 40.68 / 40.08 / 19.24;
>   `wh_owner_type` is nearly even at 54.31 / 45.69; `WH_regional_zone` ranges from Zone 6 at 33.36%
>   down to Zone 1 at 8.22%; and the five certificate grades are spread evenly between 16.76% and
>   22.00%, with no dominant grade.
>
> - The rare-level scan returns only two entries — `zone == East` and the certificate's `(missing)`
>   level — which confirms the categorical structure is otherwise sound.

> **Decision opened.**
>
> - Decide whether `zone == East` is kept as its own level or grouped.
> - Decide the encoding scheme for each categorical column:
>   - one-hot for nominal features;
>   - explicit ordering for ordinal features.
> - Defer the final choice to each objective's transformation notebook, because the right treatment differs by algorithm.


---
## 8. Outlier analysis

This section applies *"IQR and other statistical methods to identify extreme values and decide
whether to include or exclude them."* Both the IQR rule and the z-score rule are applied.

One caution governs how the result is read. The IQR rule flags anything outside
`Q1 − 1.5×IQR … Q3 + 1.5×IQR`, which assumes a roughly continuous, roughly symmetric variable. On a
0/1 indicator, or on a count taking only a handful of values, the rule flags the *rare but entirely
real* level — that is a property of the rule, not a defect in the data. So the nature of each column
is derived from its own values first, and reported alongside the flag counts. A high flagged
percentage is a prompt to look, never on its own a reason to act.


In [ ]:
rows = []
for c in num_cols:
    s = df[c].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_below, n_above = int((s < lo).sum()), int((s > hi).sum())
    n_z = int((((s - s.mean()) / s.std(ddof=0)).abs() > 3).sum())

    k = s.nunique()
    whole = bool((s % 1 == 0).all())
    nature = "binary" if k == 2 else ("count" if (k <= 45 and whole) else "continuous")

    rows.append({
        "column": c, "nature": nature, "n_unique": k,
        "min": s.min(), "Q1": q1, "median": s.median(), "Q3": q3, "max": s.max(),
        "lower_fence": round(lo, 2), "upper_fence": round(hi, 2),
        "n_below_fence": n_below, "n_above_fence": n_above,
        "pct_IQR": round(100 * (n_below + n_above) / len(s), 2),
        "n_abs_z_over_3": n_z, "pct_z": round(100 * n_z / len(s), 2),
    })

outlier_scan = pd.DataFrame(rows).sort_values("pct_IQR", ascending=False).reset_index(drop=True)
outlier_scan

> **Interpretation.**
>
> - Read this table alongside the `nature` column. Reading straight down
>   `pct_IQR` gives the wrong answer.
>
> - **The three highest flag rates are artefacts of applying the rule to the wrong kind of variable.**
>
> - `transport_issue_l1y` — 11.77% flagged, **all 2,943 above** the fence of 2.5. Its median and Q1
>   are both 0, which drags the fence down so that every warehouse reporting 3 or more transport
>   issues is flagged — precisely the warehouses this project exists to study.
> - `flood_impacted` (2,454 flagged, 9.82%) and `flood_proof` (1,366, 5.46%) — 0/1 indicators. With
>   Q1 = Q3 = 0 the IQR is zero, the fence collapses to `[0, 0]`, and **every single 1 is flagged**.
>   The z-score rule flags the same rows, because a rare 1 sits more than three standard deviations
>   from a mean close to zero. Both rules break on indicators; acting on either would delete the
>   entire minority class.
>
> - **Two continuous columns carry meaningful flag rates, with extremes at both ends:**
>
> | Column | Below the lower fence | Above the upper fence | Median |
> |---|---|---|---|
> | `retail_shop_num` | 81 rows below 2,532.5 (minimum 1,821) | 867 rows above 7,280.5 (maximum 11,008) | 4,859 |
> | `workers_num` | 5 rows below 10.5 (minimum 10) | 602 rows above 46.5 (maximum 98) | 28 |
>
> - Nothing at either end is physically implausible. A warehouse area served by 1,821 retail shops or
>   by 11,008, or a warehouse staffed by 10 workers or by 98, describes a small or a large operation,
>   not an entry error. The lower-end flags for `workers_num` are only 5 rows at the value 10, one
>   worker below the fence.
>
> - `Competitor_in_mkt` flags 96 rows (0.38%), all above the fence of 7, up to a maximum of 12.
>
> - **Ten of the sixteen columns flag nothing at all**, including both targets.
>
> - On the count columns the z-score rule is far more conservative than IQR (`transport_issue_l1y`:
>   1.39% against 11.77%). That gap is itself evidence that the IQR figure is inflated by the shape of a
>   count variable rather than by genuine extremes.
>
> - *A caveat on the `nature` column:* it is a heuristic based on the number of distinct values,
>   offered as a reading aid. It labels `wh_est_year` a "count" because years are whole numbers, and
>   labels `workers_num` and `distributor_num` "continuous" only because they exceed 45 distinct
>   values. The judgements above rest on what each variable means, not on that label.

> **Decision opened.**
>
> - Give each flagged column its own verdict: retain, cap or remove.
> - Argue the verdict individually in `01_global_preprocessing.ipynb`.
> - Do not act on a flag rate by itself.


---
## 9. The two target variables

`wh_breakdown_l3m` is the Objective 2 target and `product_wg_ton` the Objective 3 target. Their
shape determines how each problem is posed and evaluated.

For Objective 2, the target is binary: warehouses with 0–3 breakdowns in three months are classed
as **Not High Risk**; those with 4–6 breakdowns as **High Risk**. The threshold is placed at 4+
breakdowns because that is where warehouses become meaningfully different in operations — the η²
analysis that identifies this step is in the classification EDA notebook (NB 21). This section checks
the observed distribution of the count to motivate that threshold choice.


In [ ]:
t = "wh_breakdown_l3m"
vc = df[t].value_counts().sort_index()
print(f"=== {t} ===")
print(pd.DataFrame({"n": vc, "pct_of_rows": (100 * vc / len(df)).round(2)}).to_string())
print(f"\nmean {df[t].mean():.3f} | median {df[t].median():.1f} | "
      f"sd {df[t].std():.3f} | skew {df[t].skew():.3f}")

> **Interpretation.**
>
> The breakdown count spans 0 to 6 with mean 3.482 and near-zero skew, so the distribution is
> roughly symmetric around its centre. The value 0 appears only 908 times (3.63%) — far fewer than
> any other level — while levels 1 through 6 are more evenly spread. This asymmetry at zero is
> notable: a naive binary split on “any breakdown?” produces a **96.4% / 3.6% imbalance** that is
> trivially predictable from `storage_issue == 0` (see §10), not a useful classifier. The threshold
> must be placed elsewhere in the count range.

In [ ]:
# The binary banding used by Objective 2, checked against the actual distribution.
bands = pd.cut(df[t], bins=[-0.5, 3.5, 6.5],
               labels=["Not High Risk (0-3)", "High Risk (4-6)"])
band_counts = bands.value_counts().reindex(["Not High Risk (0-3)", "High Risk (4-6)"])

print("binary banding used by Objective 2:")
print(pd.DataFrame({"n": band_counts,
                    "pct_of_rows": (100 * band_counts / len(df)).round(2)}).to_string())
print(f"\nimbalance ratio, larger : smaller = "
      f"{band_counts.max() / band_counts.min():.2f} : 1")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(x=df[t], ax=axes[0])
axes[0].axvline(3.5, color="#e34948", linestyle="--", linewidth=2)
axes[0].set_title(f"{t} — raw count, cut at 4+", fontweight="bold")
sns.countplot(x=bands, order=band_counts.index, ax=axes[1])
axes[1].set_title("binary banding used by Objective 2", fontweight="bold")
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> Cutting at 4+ breakdowns gives **13,026 Not High Risk (52.10%)** against **11,974 High Risk
> (47.90%)** — a ratio of 1.09 : 1. That is close enough to balanced that no resampling is needed,
> and it is a far more usable split than the 96.4 / 3.6 the naive "any breakdown" framing produces.
>
> The threshold is not chosen for balance alone. NB 21 tests every adjacent cut point in the 0–6
> range. Two steps are larger than 3|4 — the ones at 0|1 and 1|2 — but both sit on the ramp out of
> the 908 newly commissioned warehouses described in §10, and a cut there keeps **0% of the signal
> among rated warehouses**. It separates warehouses that have not started operating from those that
> have, which is a commissioning status rather than a risk gradient.
>
> Among the cuts that actually separate operating warehouses, **3|4 is the strongest by a wide
> margin**. Balance and separation point the same way, which is why this boundary is used.

In [ ]:
t = "product_wg_ton"
print(f"=== {t} ===")
print(df[t].describe().round(2).to_string())
print(f"\nskew {df[t].skew():.3f} | kurtosis {df[t].kurtosis():.3f}")

sample = df[t].sample(5000, random_state=RANDOM_STATE)
w, p = stats.shapiro(sample)
print(f"Shapiro-Wilk on a 5,000-row sample: W = {w:.4f}, p = {p:.3g}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df[t], bins=50, kde=True, ax=axes[0]); axes[0].set_title(f"{t} distribution")
sns.boxplot(x=df[t], ax=axes[1]); axes[1].set_title("spread")
stats.probplot(sample, dist="norm", plot=axes[2]); axes[2].set_title("Q-Q against normal")
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> - **`wh_breakdown_l3m` (Objective 2).** Spans 0 to 6: mean 3.482, median 3, skew −0.068. The levels
>   are not evenly used: **only 908 warehouses (3.63%) report zero breakdowns** — by far the smallest
>   level. The next smallest is 1 breakdown at 2,036 rows (8.14%), and levels 2 to 6 each hold between
>   15.70% and 20.30%.
>
> - The naive binary framing — *did this warehouse report any breakdown?* — splits **96.37% / 3.63%**,
>   a severe imbalance. Those 908 zero-breakdown warehouses are already identifiable from
>   `storage_issue == 0` (§10), so this split is trivially predictable and carries no modelling value.
>   This motivates placing the classification threshold elsewhere in the count range.
>
> - Objective 2 uses a **binary target**: warehouses are either **Not High Risk** (0–3 breakdowns,
>   ~52.1%) or **High Risk** (4–6 breakdowns, ~47.9%). The threshold is placed at 4+ breakdowns
>   because that is the strongest single η² step across all related features (full analysis in NB 21).
>
> | Class | Breakdowns | Share |
> |---|---|---|
> | Not High Risk | 0–3 | ~52.1% |
> | High Risk | 4–6 | ~47.9% |
>
> The class balance ratio is ~1.09 : 1 — no resampling is needed.
>
> - **`product_wg_ton` (Objective 3).** Ranges from 2,065 to 55,151 t. Its summary statistics describe
>   a tidy, near-symmetric variable: mean 22,102.63 and median 22,101 within a ton of each other, skew
>   0.332, kurtosis −0.502.
>
> - **The plots show that description is incomplete.** Read from the histogram: the distribution is
>   **not a single hump**. There is a distinct low peak at roughly 5,000–7,000 t, a dip near 10,000 t,
>   a broad body from about 12,000 to 32,000 t that itself has more than one peak, and a thinning tail
>   up to the maximum. The Q-Q plot agrees: at the low end the points form a flat step instead of
>   following the line, and at both ends they bend back toward the centre, meaning the values stop
>   sooner than a normal distribution’s would.
>
> - Where the low peak comes from is not settled here. The 908 warehouses identified in §10 average
>   5,430 t and fall inside it, but the peak holds considerably more warehouses than 908, so they are
>   not the whole explanation. NB 31 examines what produces the peaks.
>
> - Shapiro–Wilk rejects normality (W = 0.9707, p = 8.24e-31). With n = 5,000 that result alone would
>   be weak evidence, because the test rejects on trivially small departures; here the plots show the
>   departure is real and structural.
>
> - **What this means for a transformation.** A log or square-root transform corrects right skew. This
>   target’s skew is mild (0.332). Its actual departures from normality are several peaks and short
>   tails, and neither transform removes those. Nor does linear regression require a normally
>   distributed target; it requires well-behaved residuals, which are checked on the fitted models in
>   NB 35. **No transformation of the target is indicated at this stage.** The reason is the kind of
>   departure seen in the plots, not the skew statistic.

---
## 10. Cross-column consistency

This section asks whether any column reproduces or constrains another. It matters for two reasons.
A column that is a near-copy of another adds no information but distorts distance-based methods
in Objective 1. More seriously, a column that **pins a target to a single value** for some group of
rows makes that target trivially predictable for those rows — which inflates model scores without
the model having learned anything.

Four scans, from specific to general:

- **A** — does any category level force a target to one constant value?
- **B** — same question for the zero-valued group of every count column.
- **C** — where several such subgroups exist, do they describe *the same warehouses*? Measured with
  the Jaccard index: shared rows divided by rows in either group. A value of 1.00 means two
  conditions select an identical set of rows.
- **D** — what else distinguishes a subgroup found in C from the rest of the network? Same SMD
  measure as §5, so any description of that subgroup rests on numbers rather than guesswork.


In [ ]:
# Scan A - does any category level pin a target to a single value?
findings = []
for c in cat_cols:
    for level, g in df.groupby(c, dropna=False):
        for t in targets:
            if g[t].nunique() == 1:
                findings.append({
                    "condition": f"{c} == {'(missing)' if pd.isna(level) else level}",
                    "n_rows": len(g),
                    "target": t,
                    "is_constant_at": g[t].iloc[0],
                })
pd.DataFrame(findings) if findings else "No category level pins either target to a single value."

> **Interpretation.**
>
> Scan A finds one alarm: the 908 rows where `approved_wh_govt_certificate` is missing all record `wh_breakdown_l3m == 0`. No other category level pins a target to a single value.

In [ ]:
# Scan B - same question for the zero-valued group of every count column.
count_cols = [c for c in num_cols if df[c].nunique() <= 45]

findings = []
for c in count_cols:
    g = df[df[c] == 0]
    if len(g) == 0:
        continue
    for t in targets:
        if t != c and g[t].nunique() == 1:
            findings.append({
                "condition": f"{c} == 0",
                "n_rows": len(g),
                "target": t,
                "is_constant_at": g[t].iloc[0],
            })
pd.DataFrame(findings) if findings else "No zero-valued group pins either target to a single value."

> **Interpretation.**
>
> Scan B finds the same 908 warehouses again: they are the zero-valued group for both `storage_issue_reported_l3m` and `wh_breakdown_l3m`. No other count-column zero group constrains a target.

In [ ]:
# Scan C - do these subgroups describe the same warehouses?
groups = {}
for c in count_cols:
    idx = set(df.index[df[c] == 0])
    if 0 < len(idx) < len(df):
        groups[f"{c} == 0"] = idx
for c in missing_cols:
    groups[f"{c} is missing"] = set(df.index[df[c].isna()])

keys = list(groups)
J = pd.DataFrame(index=keys, columns=keys, dtype=float)
for a in keys:
    for b in keys:
        A, B = groups[a], groups[b]
        J.loc[a, b] = len(A & B) / len(A | B)

plt.figure(figsize=(10, 8))
sns.heatmap(J.astype(float), annot=True, fmt=".2f", cmap="rocket_r", vmin=0, vmax=1,
            annot_kws={"size": 7}, cbar_kws={"label": "Jaccard overlap"})
plt.title("Do these subgroups describe the same warehouses?\n"
          "(1.00 = identical set of rows)", fontweight="bold")
plt.tight_layout(); plt.show()

print("group sizes:")
for k in keys:
    print(f"  {k:<42} {len(groups[k]):,} rows")

# Read the strongest overlaps off the matrix rather than leaving them to the eye.
overlaps = []
for i, a in enumerate(keys):
    for b in keys[i + 1:]:
        overlaps.append({"group_a": a, "group_b": b,
                         "shared_rows": len(groups[a] & groups[b]),
                         "jaccard": round(J.loc[a, b], 4)})
overlaps = pd.DataFrame(overlaps).sort_values("jaccard", ascending=False)

print("\nstrongest overlaps between subgroups:")
print(overlaps.head(8).to_string(index=False))

identical = overlaps[overlaps["jaccard"] == 1.0]
print(f"\npairs of conditions selecting an IDENTICAL set of rows: {len(identical)}")
if len(identical):
    print(identical.to_string(index=False))

> **Interpretation.**
>
> The Jaccard scan shows that the sets of rows identified by `approved_wh_govt_certificate is missing`, `storage_issue_reported_l3m == 0` and `wh_breakdown_l3m == 0` are identical — Jaccard 1.0000 for all three pairs. These three conditions are interchangeable names for exactly the same 908 warehouses.

In [ ]:
# Scan D - profile the subgroup found in Scan C against the rest of the network.
# It is identified here by the certificate gap; Scan C showed the other two
# conditions select exactly the same rows.
no_cert = df["approved_wh_govt_certificate"].isna()
print(f"subgroup size: {int(no_cert.sum()):,} rows   rest of network: {int((~no_cert).sum()):,} rows")

# Is zero storage issues part of the same distribution as the other values?
print("\nstorage_issue_reported_l3m — frequency of its eight lowest values:")
print(df["storage_issue_reported_l3m"].value_counts().sort_index().head(8).to_string())

print(f"\nwh_est_year missing   in subgroup: {100 * df.loc[no_cert, 'wh_est_year'].isna().mean():.2f}%"
      f"   in rest: {100 * df.loc[~no_cert, 'wh_est_year'].isna().mean():.2f}%")

rows = []
for c in num_cols:
    a, b = df.loc[no_cert, c].dropna(), df.loc[~no_cert, c].dropna()
    pooled_sd = np.sqrt((a.var() + b.var()) / 2)
    rows.append({
        "column": c,
        "mean_in_subgroup": round(a.mean(), 2),
        "mean_in_rest": round(b.mean(), 2),
        "smd": round((a.mean() - b.mean()) / pooled_sd, 3),
    })
profile = pd.DataFrame(rows)
profile["abs_smd"] = profile["smd"].abs()
profile.sort_values("abs_smd", ascending=False).drop(columns="abs_smd").reset_index(drop=True)

> **Interpretation.**
>
> The profile scan shows the 908-warehouse subgroup is ordinary in size, staffing and market position but differs sharply on operational metrics: it averages about a quarter of the typical shipment volume, logs no storage issues or breakdowns, and was established on average in 2021 — thirteen years later than the rest of the network. The most consistent reading is recently commissioned warehouses not yet fully operating.

> **Interpretation.**
>
> - This is the most consequential section in the notebook. The four scans converge
>   on a single finding.
>
> - **Scan A** found one category level that pins a target: where `approved_wh_govt_certificate` is
>   missing (908 rows), `wh_breakdown_l3m` is constant at 0.
>
> - **Scan B** found the same from another direction: where `storage_issue_reported_l3m == 0` (908 rows),
>   `wh_breakdown_l3m` is again constant at 0. It also lists `Competitor_in_mkt == 0`, but that group is
>   a **single row**, so "constant" is trivially true and means nothing.
>
> - **Scan C** shows that three conditions select an **identical** set of rows — Jaccard = **1.0000**
>   for all three pairings, 908 shared rows each time:
>
> ```
> approved_wh_govt_certificate is missing   ==   storage_issue_reported_l3m == 0   ==   wh_breakdown_l3m == 0
> ```
>
> - **Scan D shows who these 908 warehouses are.** Compared with the other 24,092:
>
> | | Subgroup | Rest | SMD |
> |---|---|---|---|
> | establishment year | **2,021.94** | 2,008.91 | **+2.522** |
> | shipment (t) | **5,430.47** | 22,730.99 | −2.099 |
> | temperature regulating machine | **0.01** | 0.31 | −0.890 |
> | transport issues | 0.62 | 0.78 | −0.138 |
>
> - On **every other column** — workers, distributors, retail shops, distance from hub, competitors,
>   refill requests, flood exposure, electric back-up and government checks — they are
>   indistinguishable from the rest, with every |SMD| at or below 0.096.
>
> - The year average rests on the roughly half of the subgroup that has a recorded year: 47.91% of
>   them are missing it, essentially the same rate as the rest of the network (47.51%). So these
>   warehouses are no worse recorded than any others; they are simply newer. For comparison, §6 shows
>   the latest establishment year anywhere in the data is 2023.
>
> - Scan D also prints the lowest values of `storage_issue_reported_l3m`: **0 occurs 908 times, the
>   values 1, 2 and 3 never occur, and the next value is 4 (1,081 rows).** Zero is separated from the
>   rest of the distribution by a gap, so the subgroup is a distinct population rather than the low tail
>   of a continuum.
>
> - **The finding.** 908 warehouses of ordinary size, staffing and market position, established on
>   average in 2021–2022, with no certificate issued, no storage issue or breakdown logged, almost no
>   temperature regulation, and about a quarter of the typical shipment volume. The reading most
>   consistent with the evidence is **recently commissioned warehouses not yet fully operating** —
>   not warehouses performing flawlessly. It remains an interpretation of a snapshot; the file records
>   no commissioning or go-live date to confirm it.
>
> - **Consequence for Objective 2.** A binary "any breakdown vs none" target would be **perfectly
>   predictable from one rule** — `if storage_issue_reported_l3m == 0 then no breakdown` — giving 100%
>   accuracy while learning nothing about breakdown risk. Every standard metric would read
>   1.00 and the result would be worthless. The binary banding used for Objective 2 removes the problem: these
>   908 rows sit *inside* the Low class of 8,020 rather than forming a class of their own. §9 supported
>   the banding on balance; this supports it on validity, the stronger argument.
>
> - **Consequence for Objective 3.** These rows are legitimate and stay in, but they sit at the low end
>   of shipment weight. Percentage error grows as the actual value shrinks, so they will weigh heavily
>   on MAPE — to be reported alongside that metric in NB 35, not discovered afterwards.
>
> - **One non-finding.** `flood_impacted == 0` and `flood_proof == 0` overlap at Jaccard 0.8708. That is
>   not redundancy: both indicators are rarely 1, so both zero-groups are very large and are bound to
>   overlap. A high Jaccard between two *common* conditions means much less than between two rare ones.

> **Decision opened.**
>
> - Keep these warehouses, in line with the project's stated scope.
> - Flag them rather than dropping them.
> - Use this section as the evidence for why the flag is needed.
> - Resolve the flag definition in `01_global_preprocessing.ipynb`.
> - Resolve the certificate `NA` treatment there as well.


---
## 11. Correlation overview

This section measures pairwise relationships between variables. Pearson measures straight-line association; Spearman measures whether the ranks move
together, which catches monotonic-but-curved relationships Pearson would understate. Reporting both
shows whether a weak Pearson value means *no relationship* or merely *not a straight line*.

How the collinearity found here is handled is left to each objective — a distance-based clustering
model and a tree-based regressor are affected quite differently.


In [ ]:
pearson = df[num_cols].corr(method="pearson")
spearman = df[num_cols].corr(method="spearman")

plt.figure(figsize=(13, 10))
sns.heatmap(pearson, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
            annot_kws={"size": 7}, cbar_kws={"label": "Pearson r"})
plt.title("Pearson correlation — numeric columns", fontweight="bold")
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> The heatmap reveals one dominant block coloured at the deep end of the scale: `storage_issue_reported_l3m`, `product_wg_ton` and `wh_est_year` are mutually correlated well above 0.80. Outside this block, no pair of columns reaches 0.35.

In [ ]:
mask = np.triu(np.ones(pearson.shape), k=1).astype(bool)
pairs = pearson.where(mask).stack().reset_index()
pairs.columns = ["column_a", "column_b", "pearson_r"]
pairs["spearman_rho"] = [spearman.loc[a, b] for a, b in zip(pairs.column_a, pairs.column_b)]
pairs["abs_r"] = pairs["pearson_r"].abs()

print("15 strongest pairwise relationships:")
pairs.sort_values("abs_r", ascending=False).head(15).drop(columns="abs_r").round(3).reset_index(drop=True)

> **Interpretation.**
>
> - The correlation structure is dominated by one tightly-coupled block of three
>   variables, with almost nothing else of consequence.
>
> - **The block.**
>
> | Pair | Pearson | Spearman |
> |---|---|---|
> | `storage_issue_reported_l3m` ↔ `product_wg_ton` | **0.987** | 0.989 |
> | `wh_est_year` ↔ `storage_issue_reported_l3m` | **−0.859** | −0.872 |
> | `wh_est_year` ↔ `product_wg_ton` | **−0.829** | −0.850 |
>
> - A Pearson correlation of 0.987 is close to a deterministic relationship: the Objective 3 target is
>   very nearly a straight-line function of a single predictor. Since `wh_est_year` runs backwards
>   (a smaller year means an older warehouse), the three read as one story — **older warehouses report
>   more storage issues and ship more product**.
>
> - The preliminary analysis supports a direct association: **shipment weight moves mainly with reported
>   storage issues and warehouse age**. Causation still cannot be claimed from a single snapshot.
>
> - **The second tier is far weaker**: `wh_est_year` ↔ `wh_breakdown_l3m` (−0.399),
>   `storage_issue_reported_l3m` ↔ `wh_breakdown_l3m` (0.377), `wh_breakdown_l3m` ↔ `product_wg_ton`
>   (0.343). Modest but not negligible — and notably, the Objective 2 target's three strongest
>   relationships are all with members of that same block.
>
> - **Everything else is weak.** The strongest relationship not involving the block is
>   `electric_supply` ↔ `workers_num` at 0.340, then `num_refill_req_l3m` ↔ `temp_reg_mach` at 0.261.
>   All remaining pairs sit below 0.18.
>
> - **Pearson and Spearman agree closely throughout** — 0.987 against 0.989, −0.859 against −0.872,
>   and so on down the table. That agreement is informative in itself: it means the weak values are
>   genuinely weak *relationships*, not strong curved ones that Pearson is understating. A
>   straight-line measure and a rank-based measure reaching the same conclusion is good evidence that
>   most of these features simply do not move together.

> **Decision opened.**
>
> - The collinear block needs handling.
> - The right handling differs by algorithm:
>   - distance-based clustering would triple-count one underlying dimension;
>   - regularised regression and tree models can handle the block differently.
> - Defer the Objective 1 treatment to NB 11/13.
> - Defer the Objective 3 treatment to NB 31/32, where VIF quantifies it properly.


---
## 12. Confirmation that nothing was modified

This notebook is an audit. It must leave the data exactly as it found it: the reload below must match
the dataframe held in memory, and nothing in `data/preprocessed/` may have been created or changed
while it ran. (That folder is written by `01_global_preprocessing.ipynb`, so it may already hold a
file — the check compares against a snapshot taken in §0, not against an empty folder.)


In [ ]:
assert df.equals(load_raw()), "df was modified — this notebook must not alter the data"

preprocessed_now = {p.name: p.stat().st_mtime for p in PREPROCESSED_DIR.glob('*')}
assert preprocessed_now == preprocessed_at_start, "this notebook wrote to data/preprocessed/"

print("raw dataframe unchanged                 : True")
print(f"shape still                             : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"data/preprocessed/ untouched by this run: True "
      f"({len(preprocessed_now)} file(s) present, none created or changed here)")

---
## Summary — what this notebook established

**The dataset is structurally clean.** 25,000 rows × 24 columns as specified; no duplicate records of
any kind; no negative or `-1`-style sentinels; every field stored in a type consistent with its
business definition; gaps spread evenly through the file rather than concentrated in one batch.

**Three columns have gaps. They are three different problems, and none is purely random.**

| Column | Gap | What the audit showed |
|---|---|---|
| `wh_est_year` | 11,881 (47.52%) | Flat across every categorical, but rows without a year have a distinct operating profile — above all, fewer than half the refill requests (2.55 vs 5.49, SMD −1.353). |
| `workers_num` | 990 (3.96%) | Unrelated to either target, but linked to site conditions — more often flood-impacted (SMD +0.391), flood-proof (+0.368) and with electric back-up (+0.190). |
| `approved_wh_govt_certificate` | 908 (3.63%) | Not a gap. The cells literally contain the text `'NA'`, and the group differs from the rest by two to three standard deviations on several columns. |

**The central finding is structural.** Three conditions — certificate `NA`, `storage_issue == 0` and
`breakdown == 0` — select an **identical** set of 908 warehouses (Jaccard = 1.0000). They match the
rest of the network on size, staffing and market, but were established on average in 2021.94 against
2008.91, ship about a quarter of the typical volume, and almost none have temperature regulation. The
most consistent reading is recently commissioned warehouses not yet fully operating. Their existence
would have made a binary breakdown target 100% predictable from one rule, which independently
justifies the three-class banding.

**Summary statistics hid shape more than once.** `product_wg_ton` has skew 0.332 and near-identical
mean and median, but its histogram has several peaks and its Q-Q plot shows short tails and a step at
the low end. `storage_issue_reported_l3m` jumps from 0 straight to 4, and `govt_check_l3m` is strongly
spiked. In each case a mean describes few real warehouses.

**The correlation structure is dominated by one block.** `storage_issue_reported_l3m`,
`product_wg_ton` and `wh_est_year` are mutually correlated at 0.83 to 0.99. No pair of columns outside
that block exceeds 0.34. This supports the stated project expectation that shipment weight is driven
mainly by storage issues and warehouse age — as an association, not a proven cause.

**Both targets suit their objectives without transformation.** The three-class breakdown banding
lands at 32.08 / 36.17 / 31.75% (ratio 1.14 : 1). The shipment target departs from normality through
multiple peaks and short tails, which a log or square-root transform would not fix.

**Outlier flags need reading with care.** The three highest IQR flag rates come from applying a
continuous-variable rule to counts and 0/1 indicators. Only `retail_shop_num` (81 low, 867 high) and
`workers_num` (5 low, 602 high) are genuine continuous candidates, and neither end is implausible.

**Nothing was modified.** The final check confirms the dataframe still matches a fresh read of the
source file, and that this notebook created or changed nothing in `data/preprocessed/`.

---

### Open questions for the preprocessing notebook

Several decisions follow from the evidence above and are worked through in the next notebook. The
key ones are:

- **The identifier columns.** Both `Ware_house_ID` and `WH_Manager_ID` are one per row, so neither
  is useful as a model feature. The question is what to keep as a row key.
- **The certificate's literal `'NA'`.** These 908 cells are a recorded state, not a gap, and the
  group they mark is distinctive enough to deserve its own treatment.
- **The `wh_est_year` gap (47.52%).** Three options are worth measuring — dropping the rows,
  filling with the median, or predicting from related columns. §5 and §11 provide the evidence;
  §4 shows the gap follows other columns in a structured way.
- **The `workers_num` gap (3.96%).** The gap is unrelated to the targets but linked to site
  conditions, so a group-specific fill is worth checking.
- **Integer types.** `workers_num` and `wh_est_year` are stored as `float64` only because they
  held gaps; once filled they should be restored to integers.
- **Outliers.** §8 flags several columns, but read alongside the `nature` column the picture is
  less alarming than the raw counts suggest.
- **Row removal.** §3 found no duplicates, and §10 established the 908 unusual warehouses are a
  coherent subgroup rather than errors.

Two further decisions depend on the algorithm and are deferred to each objective's own notebooks:
encoding categorical columns, and how to handle the collinear block (r = 0.83 to 0.99) identified
in §11.